# XAI - Learned per-signature localizer (Analysis 4): is the signal localizable?

The first three analyses used the attribution of the **frozen model** (gradients) and localized
nothing, even though signal was present at the feature level (Analysis 3: AUC up to 0.83). The
question here:

> If we train a **per-signature attention-MIL** that is *free to learn where to look*, can it (a)
> predict the signature **better** than the slide mean and the gradient attribution, and (b)
> **concentrate** the signal on signature-specific patches?

For each (tissue, signature) we train a gated ABMIL (Ilse 2018) on patch bags (1000 patches per
slide), plus/minus labels, **5-fold CV by patient**. From the OOF attention we then take the top
patches and measure:
- **ABMIL AUC** against the baselines (slide mean, grad-topX from Analysis 3);
- **concentration**: do the top-attention patches predict the signature better than the slide mean?
- **specificity**: does the attention learned for different signatures localize on different
  patches (low Jaccard)?

If ABMIL clearly beats the slide mean and the attention concentrates and specializes, the signal
**is localizable** and the limitation was the gradient attribution. If ABMIL is on par with the
slide mean, the signal really is **diffuse**. GPU-free (CPU torch).


## 0. Config

In [ ]:
import os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn
import warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

OUT_DIR='../grad/xai_outputs'; SPLIT_DIR='../coad_stad_explainability_4fold'
TISSUES={'coad':['SBS18','SBS44','SBS93'],'stad':['SBS8','SBS18','SBS40a','SBS44']}
M=1000; H=128; EPOCHS=25; LR=1e-3; WD=1e-4; KTOP=100; N_SPLITS=5; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
print('M=',M,'H=',H,'epochs=',EPOCHS,'Ktop=',KTOP)

## 1. Modello ABMIL + training CV (con attention out-of-fold)

In [ ]:
def load_bags(tissue):
    d=np.load(os.path.join(OUT_DIR,f'bags_{tissue}_M{M}.npz'),allow_pickle=True)
    return d['bags'], np.array([str(x) for x in d['case_id']])

class ABMIL(nn.Module):
    def __init__(self,d=1536,h=H,p=0.25):
        super().__init__()
        self.enc=nn.Sequential(nn.Linear(d,h),nn.ReLU(),nn.Dropout(p))
        self.V=nn.Linear(h,h); self.U=nn.Linear(h,h); self.w=nn.Linear(h,1); self.head=nn.Linear(h,1)
    def forward(self,x):
        H_=self.enc(x)
        A=torch.softmax(self.w(torch.tanh(self.V(H_))*torch.sigmoid(self.U(H_))).squeeze(-1),dim=1)
        z=(A.unsqueeze(-1)*H_).sum(1)
        return self.head(z).squeeze(-1), A

def labels_for(tissue,target,cases):
    sp=pd.read_csv(os.path.join(SPLIT_DIR,f'{tissue}_{target}.csv'))
    lab=dict(zip(sp['Case ID'],(sp['group']==f'{target}_plus').astype(int)))
    return np.array([lab.get(c,0) for c in cases])

def cv_train(tissue,target,bags,cases,seed=SEED):
    '''CV per paziente; ritorna AUC a livello caso, predizioni OOF e attention OOF [n,M].'''
    y=labels_for(tissue,target,cases)
    ucase=np.array(sorted(set(cases))); cl=np.array([y[list(cases).index(c)] for c in ucase])
    skf=StratifiedKFold(N_SPLITS,shuffle=True,random_state=seed)
    oof=np.full(len(y),np.nan); oofA=np.zeros((len(y),M),dtype=np.float32)
    for tr,te in skf.split(ucase,cl):
        trc=set(ucase[tr]); tem=np.array([c in set(ucase[te]) for c in cases]); trm=np.array([c in trc for c in cases])
        Xtr=torch.tensor(bags[trm].astype(np.float32)); ytr=torch.tensor(y[trm].astype(np.float32)); Xte=torch.tensor(bags[tem].astype(np.float32))
        pos=max(ytr.sum().item(),1); negw=(len(ytr)-pos)/pos
        m=ABMIL(); opt=torch.optim.Adam(m.parameters(),lr=LR,weight_decay=WD)
        lf=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(negw))); m.train()
        for ep in range(EPOCHS):
            perm=torch.randperm(len(Xtr))
            for i in range(0,len(Xtr),16):
                bi=perm[i:i+16]; opt.zero_grad(); lo,_=m(Xtr[bi]); lf(lo,ytr[bi]).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            lo,A=m(Xte); oof[tem]=lo.numpy(); oofA[tem]=A.numpy()
    dfp=pd.DataFrame({'case':cases,'p':oof,'y':y}).groupby('case').agg({'p':'mean','y':'first'})
    return roc_auc_score(dfp['y'],dfp['p']), oof, oofA, y

def probe_auc(X,y,seed=SEED):
    if len(np.unique(y))<2 or min(np.bincount(y))<N_SPLITS: return np.nan
    npca=int(min(50,X.shape[0]-N_SPLITS,X.shape[1]))
    pipe=make_pipeline(StandardScaler(),PCA(npca,random_state=seed),LogisticRegression(max_iter=1000,class_weight='balanced'))
    return float(cross_val_score(pipe,X,y,cv=StratifiedKFold(N_SPLITS,shuffle=True,random_state=seed),scoring='roc_auc').mean())

## 2. Train, compare against the baselines, extract attention
For each target: ABMIL AUC (CV) against slide-mean and grad-topX (Analysis 3). From the OOF
attention: top-KTOP patches -> mean feature -> probe (concentration), with the top indices saved
for the cross-signature specificity check.


In [ ]:
base=pd.read_csv(os.path.join(OUT_DIR,'decodability_ALL.csv')) if os.path.exists(os.path.join(OUT_DIR,'decodability_ALL.csv')) else None
rows=[]; ATT_TOP={}
for tissue,targets in TISSUES.items():
    bags,cases=load_bags(tissue); t0=time.time()
    for target in targets:
        auc,oof,oofA,y=cv_train(tissue,target,bags,cases)
        # concentration: mean feature of the top-KTOP patches by attention
        top=np.argsort(-oofA,axis=1)[:,:KTOP]                       # [n,KTOP]
        ATT_TOP[(tissue,target)]=top
        attfeat=np.stack([bags[i,top[i]].astype(np.float32).mean(0) for i in range(len(bags))])
        auc_attfeat=probe_auc(attfeat,y)
        slm=gtx=np.nan
        if base is not None:
            r=base[(base['tissue']==tissue)&(base['target']==target)]
            if len(r): slm=float(r['AUC_slide_mean'].iloc[0]); gtx=float(r['AUC_topX'].iloc[0])
        rows.append({'tissue':tissue,'target':target,'n_plus':int(y.sum()),'n_minus':int((y==0).sum()),
                     'AUC_ABMIL':round(auc,3),'AUC_attn_topK':round(auc_attfeat,3),
                     'AUC_slide_mean':round(slm,3),'AUC_grad_topX':round(gtx,3),
                     'gain_vs_mean':round(auc-slm,3) if slm==slm else np.nan})
        print(f'{tissue} {target}: ABMIL={auc:.3f} attnTopK={auc_attfeat:.3f} mean={slm} gradTopX={gtx}  t={time.time()-t0:.0f}s',flush=True)
res=pd.DataFrame(rows); res.to_csv(os.path.join(OUT_DIR,'abmil_localizer_ALL.csv'),index=False)
res

## 3. Specificita' dell'attention tra firme (Jaccard dei top-attention patch)

In [ ]:
import itertools
spec=[]
for tissue,targets in TISSUES.items():
    avail=[t for t in targets if (tissue,t) in ATT_TOP]
    if len(avail)<2: continue
    n=ATT_TOP[(tissue,avail[0])].shape[0]
    for a,b in itertools.combinations(avail,2):
        Ta=ATT_TOP[(tissue,a)]; Tb=ATT_TOP[(tissue,b)]
        js=[len(set(Ta[i])&set(Tb[i]))/len(set(Ta[i])|set(Tb[i])) for i in range(n)]
        spec.append({'tissue':tissue,'pair':f'{a}-{b}','mean_jaccard_attn':round(float(np.mean(js)),3)})
spec=pd.DataFrame(spec)
print('Jaccard medio tra top-attention di firme diverse (basso = attention firma-specifica):')
spec

## 4. Lettura
- **AUC_ABMIL >> AUC_slide_mean** (gain_vs_mean alto) -> un localizzatore appreso estrae segnale che lo
  slide-mean e la grad-attribution non catturano: il segnale **e' localizzabile**.
- **AUC_ABMIL ~ AUC_slide_mean** -> il segnale e' diffuso, neanche un modello dedicato lo concentra.
- **AUC_attn_topK > AUC_slide_mean** -> le patch ad alta attention appresa sono davvero piu' informative
  (concentrazione reale), a differenza delle top grad (Analisi 3).
- **Jaccard attention basso tra firme** -> l'attention appresa e' firma-specifica (si localizza in punti
  diversi), il contrario dell'attention condivisa del modello congelato.
- Confronto diretto col grad: `AUC_ABMIL`/`AUC_attn_topK` vs `AUC_grad_topX`.

Caveat: ABMIL piccolo su CPU, M=1000 patch sottocampionate, n+ piccoli per SBS93/SBS8; CV per paziente
ma stesso fold-seed -> stime indicative. Se ABMIL vince, vale la pena scalare (piu' patch, piu' epoche).